In [0]:
from pyspark.sql.functions import col, row_number, count, lit
from pyspark.sql.window import Window 
nss_themes = spark.table("nss_themes")
people_promise = spark.table("silver_people_promise")
metric_table = spark.table("metric_dimension")
staff_table = spark.table("staff_dimension")
org_table = spark.table("org_dimension")
region_table = spark.table("region_dimension")

In [0]:
row_window = Window.orderBy(
    col("region_key"),
    col("metric_code"),
    col("org_key")
)

themes_scores = (
    nss_themes.alias("n")

    .join(
        region_table.alias("r"),
        col("n.region") == col("r.region_name"),
        how="left"
    )

    .join(
        metric_table.alias("m"),
        col("n.metric_code") == col("m.metric_code"),
        how="left"
    )

    .join(
        org_table.alias("o"),
        col("n.org_name") == col("o.org_name"),
        how="left"
    )

    .select(
        col("r.region_key").alias("region_key"),
        col("m.metric_code").alias("metric_code"),
        col("o.org_key").alias("org_key"),
        col("n.score").alias("score"),
        col("n.total_responses").alias("total_responses")
    )
)

people_promise_scores = (
    people_promise.alias("p")

    .join(
        region_table.alias("r"),
        col("p.region") == col("r.region_name"),
        how="left"
    )

    .join(
        metric_table.alias("m"),
        col("p.metric_code") == col("m.metric_code"),
        how="left"
    )

    .join(
        org_table.alias("o"),
        col("p.org_name") == col("o.org_name"),
        how="left"
    )

    .select(
        col("r.region_key").alias("region_key"),
        col("m.metric_code").alias("metric_code"),
        col("o.org_key").alias("org_key"),
        col("p.score").alias("score"),
        col("p.total_responses").alias("total_responses")
    )
)

fact_staff_survey_gold = (
    themes_scores
    .unionByName(people_promise_scores)

    .withColumn(
        "survey_key",
        row_number().over(row_window)
    )

    .select(
        "survey_key",
        "region_key",
        "metric_code",
        "org_key",
        "score",
        "total_responses"
    )
 
)



fact_staff_survey_gold.where(col("region_key").isNull()).show(10)




In [0]:

    (fact_staff_survey_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/fact_survey_scores/"
    ) \
    .saveAsTable(
        "fact_staff_survey_gold"
    ))